# Align Development Notebook

This notebook is for interactive experiments with `vg_pipeline.align`. Use it to validate parsing, depth sampling, and back-projection logic before migrating changes into the main codebase.

In [ ]:
import os
import sys
from pathlib import Path

root = Path('..').resolve()
sys.path.append(str(root))
print('Project root:', root)

from PIL import Image
from vg_pipeline.io import load_observation_npy
from vg_pipeline.align import (
    parse_align_result,
    parse_align_results_multi,
    sample_depth_median,
    deproject_pixel,
    grasp_scale_anchor,
)
from vg_pipeline.providers import run_vg_inference
from vg_pipeline.prompting import build_align_prompt_multi
from grasp_server.align_grasp import (
    _pose_from_point_and_angle,
    build_align_grasp,
    _DEFAULT_WIDTH_M,
)
from grasp_server.grasp_selection import _rotation_to_quaternion_xyzw

print(
    'Imported project helpers: load_observation_npy, vg_pipeline.align, Gemini inference,\n'
    '_pose_from_point_and_angle, _rotation_to_quaternion_xyzw, build_align_grasp'
)

## Part 1: Data Preparation
Load or create sample RGB, depth, and camera intrinsics for testing the alignment pipeline.

In [2]:
import numpy as np
import json

# Load sample RGBD data from captured images using the project loader
# (same helper the server uses: vg_pipeline.io.load_observation_npy)
sample_dir = root / 'rgbd_data' / 'captures' / '20260417_115700'
camera_data = load_observation_npy(sample_dir / 'camera_data.npy')

# Extract RGB, depth, and camera intrinsics
rgb_image = camera_data['rgb'].astype(np.uint8)
depth_map = camera_data['depth'].astype(np.float32)  # in meters
K = camera_data['K'].astype(np.float32)  # camera intrinsic matrix 3x3

print(f"RGB shape: {rgb_image.shape}")
print(f"Depth shape: {depth_map.shape}")
print(f"Camera intrinsics K:\n{K}")
print(f"Depth value range: [{np.nanmin(depth_map):.3f}, {np.nanmax(depth_map):.3f}] m")


RGB shape: (720, 1280, 3)
Depth shape: (720, 1280)
Camera intrinsics K:
[[921.9345    0.      638.43256]
 [  0.      922.3778  365.15457]
 [  0.        0.        1.     ]]
Depth value range: [0.000, 2.568] m


In [3]:
import re

# Gemini/VLM API key: prefer the environment, else read it from the zsh rc files (macOS).
def _api_key_from_zsh_rc():
    for rc in ('.zshrc', '.zshenv', '.zprofile'):
        path = Path.home() / rc
        if not path.exists():
            continue
        m = re.search(r'export\s+(GEMINI_API_KEY|GOOGLE_API_KEY)=["\']?([^"\'\n]+)', path.read_text())
        if m:
            return m.group(1), m.group(2)
    return None, None

api_key = os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY')
api_key_name = 'GEMINI_API_KEY' if os.environ.get('GEMINI_API_KEY') else 'GOOGLE_API_KEY' if api_key else None
if api_key is None:
    api_key_name, api_key = _api_key_from_zsh_rc()
    if api_key:
        os.environ[api_key_name] = api_key
        print(f'Loaded {api_key_name} from shell rc file.')

if api_key is None:
    raise RuntimeError('Set GEMINI_API_KEY or GOOGLE_API_KEY (e.g. export it in ~/.zshrc).')

print('Using API key from env var:', api_key_name)


Loaded GOOGLE_API_KEY from shell rc file.
Using API key from env var: GOOGLE_API_KEY


## Part 2: Gripper-aware VLM Inference & Parse

The Vision Language Model (VLM) returns a JSON with a normalized alignment point [y, x]
(0-1000 scale) and a `gripper_angle_deg`. To make the **angle** sensible we make the VLM
gripper-aware: the prompt now carries a parallel-jaw **gripper specification** (max opening
80 mm, min 20 mm) plus a **scale anchor** computed from depth + intrinsics via
`grasp_scale_anchor` — it expresses those metric openings in pixels at the scene's
representative depth, so the model can judge whether the object's width along the closing
direction actually fits the jaw. This is a feasibility constraint on the angle only; no
metric width is added to the output. We then parse the result to full-resolution pixels.

In [ ]:
# Real Gemini/VLM inference using project logic.
# The API key is loaded in Part 1 (api_key / api_key_name).

provider = 'gemini'
task_spec = 'the rail'
num_candidates = 1

# Scale anchor: map the gripper's metric opening limits (80 mm / 20 mm) to pixel sizes at
# the scene's representative depth, so the prompt's feasibility check is grounded in this
# image's scale. Same helper the server uses in grasp_server.app._run_align_pipeline.
z_ref, max_open_px, min_open_px = grasp_scale_anchor(depth_map, K)
print(f'Scale anchor: z_ref={z_ref:.3f} m  max_open={max_open_px:.0f} px  min_open={min_open_px:.0f} px')

prompt = build_align_prompt_multi(
    task_spec,
    w=rgb_image.shape[1],
    h=rgb_image.shape[0],
    num_candidates=num_candidates,
    max_open_px=max_open_px,
    min_open_px=min_open_px,
    ref_depth_m=z_ref,
)

print('Using task spec:', task_spec)
print('Using provider:', provider)
print('Using API key from env var:', api_key_name)
print('Prompt preview:\n', prompt[:1000], '...\n')

raw_model_text = run_vg_inference(
    provider=provider,
    images=[Image.fromarray(rgb_image)],
    task_spec=task_spec,
    model_path='gemini-robotics-er-1.6-preview',
    num_candidates=num_candidates,
    api_key=api_key,
    openai_image_mime_types=['image/png'],
    gemini_image_mime_types=['image/png'],
    prompt=prompt,
)

print('Raw Gemini response:\n')
print(raw_model_text)

parsed_results = parse_align_results_multi(
    raw_model_text,
    canvas_h=rgb_image.shape[0],
    canvas_w=rgb_image.shape[1],
    rgb_h=rgb_image.shape[0],
)
if not parsed_results:
    raise RuntimeError('No align results returned from Gemini parse.')

align_result = parsed_results[0]
print(f'\nParsed Align Result:')
print(f'  Point (pixel coordinates [y, x]): {align_result.point_yx}')
print(f'  Angle (degrees): {align_result.angle_deg}')
print(f'  Width (meters): {align_result.width_m}')

## Part 3: Sample Depth at Alignment Point
Extract the depth value at (or near) the alignment point. We take the median of valid depth values
in a 5×5 window around the point to be robust to noise and missing data.

In [5]:
v, u = align_result.point_yx  # pixel coordinates
print(f"Sampling depth at pixel ({v}, {u})")

# Sample depth using a 5×5 window around the point
depth_window = 5
z_m = sample_depth_median(depth_map, v, u, window=depth_window)

print(f"Sampled depth (median in {depth_window}×{depth_window} window): {z_m:.4f} m")
print(f"Depth in cm: {z_m * 100:.2f} cm")


Sampling depth at pixel (270, 753)
Sampled depth (median in 5×5 window): 0.5820 m
Depth in cm: 58.20 cm


## Part 4: Back-project 2D Pixel to 3D Camera Coordinates
Using the pinhole camera model and intrinsic matrix K, convert the 2D pixel + depth to a 3D point
in the camera frame: (X, Y, Z) where Z is along the optical axis (approach direction).

In [6]:
# Back-project pixel (u, v) with depth z to camera-frame 3D point
# Using pinhole camera model: [X, Y, Z] = [(u - cx) * Z / fx, (v - cy) * Z / fy, Z]

position_xyz = deproject_pixel(u, v, z_m, K)

print(f"Back-projection from pixel ({u}, {v}) with depth {z_m:.4f} m:")
print(f"  3D Position (camera frame):")
print(f"    X: {position_xyz[0]:.4f} m")
print(f"    Y: {position_xyz[1]:.4f} m")
print(f"    Z: {position_xyz[2]:.4f} m (approach direction)")
print(f"  Distance from camera: {np.linalg.norm(position_xyz):.4f} m")


Back-projection from pixel (753, 270) with depth 0.5820 m:
  3D Position (camera frame):
    X: 0.0723 m
    Y: -0.0600 m
    Z: 0.5820 m (approach direction)
  Distance from camera: 0.5895 m


## Part 5: Build 6-DoF Grasp Pose
From the 3D position and in-plane rotation angle, construct a 4×4 pose matrix with:
- X-axis (column 0): gripper closing direction (in image plane)
- Y-axis (column 1): lateral direction (cross product)
- Z-axis (column 2): approach direction (fixed to camera +Z)
- Position (column 3): the 3D point

In [7]:
# Build the 4×4 pose matrix from position and angle using the project helper
# (grasp_server.align_grasp._pose_from_point_and_angle) so the notebook exercises
# the exact production pose convention instead of re-deriving it:
#   - approach is fixed to +Z (camera optical axis)
#   - closing direction lies in the image plane, rotated by angle_deg
angle_deg = align_result.angle_deg
pose_4x4 = _pose_from_point_and_angle(position_xyz, angle_deg)

# pose columns: 0 = closing (in-plane @ angle), 1 = lateral, 2 = approach (+Z)
x_axis, y_axis, z_axis = pose_4x4[:3, 0], pose_4x4[:3, 1], pose_4x4[:3, 2]

print(f"4×4 Pose Matrix (camera frame):")
print(pose_4x4)
print(f"\nOrientation vectors:")
print(f"  X-axis (closing):  {x_axis}")
print(f"  Y-axis (lateral):  {y_axis}")
print(f"  Z-axis (approach): {z_axis}")


4×4 Pose Matrix (camera frame):
[[ 1.          0.          0.          0.07232429]
 [ 0.          1.          0.         -0.06004043]
 [ 0.          0.          1.          0.58200002]
 [ 0.          0.          0.          1.        ]]

Orientation vectors:
  X-axis (closing):  [1. 0. 0.]
  Y-axis (lateral):  [0. 1. 0.]
  Z-axis (approach): [0. 0. 1.]


## Part 6: Convert to Quaternion and Final Output
Convert the rotation matrix to a quaternion (xyzw format) for compatibility with ROS2 and downstream
processing. Assemble the final grasp dictionary with all 6-DoF information.

In [8]:
# Convert rotation matrix to quaternion (xyzw format) using the SAME converter the
# server uses (grasp_server.grasp_selection._rotation_to_quaternion_xyzw, Shepperd's
# method) rather than scipy — so this matches the production code path exactly.
R = pose_4x4[:3, :3]
qx, qy, qz, qw = _rotation_to_quaternion_xyzw(R)

print(f"Quaternion (xyzw): [{qx:.6f}, {qy:.6f}, {qz:.6f}, {qw:.6f}]")

# Assemble the final grasp dictionary by hand to document the output schema.
# NOTE: grasp_server.align_grasp.build_align_grasp() produces this exact dict in one
# call (see Part 7, which verifies the two paths agree field-for-field).
gripper_width = align_result.width_m if align_result.width_m is not None else _DEFAULT_WIDTH_M

grasp_dict = {
    "score": 1.0,
    "model_confidence": None,
    "pose_4x4": pose_4x4.tolist(),
    "position_xyz": position_xyz.tolist(),
    "quaternion_xyzw": [qx, qy, qz, qw],
    "width_m": float(gripper_width),
    "approach_dir_xyz": pose_4x4[:3, 2].tolist(),
    "source": {
        "candidate_index": 0,
        "segment_id": 0,
        "grasp_index": 0,
        "predictions_npz": None,
    },
}

print("\n=== FINAL OUTPUT: 6-DoF Grasp ===")
print(json.dumps(grasp_dict, indent=2, default=str))


Quaternion (xyzw): [0.000000, 0.000000, 0.000000, 1.000000]

=== FINAL OUTPUT: 6-DoF Grasp ===
{
  "score": 1.0,
  "model_confidence": null,
  "pose_4x4": [
    [
      1.0,
      0.0,
      0.0,
      0.07232428509296399
    ],
    [
      0.0,
      1.0,
      0.0,
      -0.060040432248501065
    ],
    [
      0.0,
      0.0,
      1.0,
      0.5820000171661377
    ],
    [
      0.0,
      0.0,
      0.0,
      1.0
    ]
  ],
  "position_xyz": [
    0.07232428509296399,
    -0.060040432248501065,
    0.5820000171661377
  ],
  "quaternion_xyzw": [
    0.0,
    0.0,
    0.0,
    1.0
  ],
  "width_m": 0.05,
  "approach_dir_xyz": [
    0.0,
    0.0,
    1.0
  ],
  "source": {
    "candidate_index": 0,
    "segment_id": 0,
    "grasp_index": 0,
    "predictions_npz": null
  }
}


## Part 7: Verify End-to-End Pipeline
Run the complete pipeline using the `build_align_grasp()` function to verify it produces the same result.

In [9]:
# Use project helper to build the final grasp(s) from the aligned point
grasps_list = build_align_grasp(
    depth_map,
    K,
    point_yx=align_result.point_yx,
    angle_deg=align_result.angle_deg,
    width_m=align_result.width_m,
    depth_window=5,
)

print("=== End-to-End Project Helper Result ===")
print(f"Number of grasps: {len(grasps_list)}")
grasp_output = grasps_list[0]
print("\nGrasp dictionary:")
print(json.dumps(grasp_output, indent=2, default=str))

# Compare with the manual build path if present
if 'grasp_dict' in globals():
    print("\n=== Verification ===")
    print(f"Position matches: {np.allclose(grasp_output['position_xyz'], grasp_dict['position_xyz'])}")
    print(f"Quaternion matches: {np.allclose(grasp_output['quaternion_xyzw'], grasp_dict['quaternion_xyzw'])}")
    print(f"Width matches: {grasp_output['width_m'] == grasp_dict['width_m']}")
    print("\n✓ Complete 2D-to-6D alignment pipeline verified!")
else:
    print("No manual grasp_dict available for direct comparison.")


=== End-to-End Project Helper Result ===
Number of grasps: 1

Grasp dictionary:
{
  "score": 1.0,
  "model_confidence": null,
  "pose_4x4": [
    [
      1.0,
      0.0,
      0.0,
      0.07232428509296399
    ],
    [
      0.0,
      1.0,
      0.0,
      -0.060040432248501065
    ],
    [
      0.0,
      0.0,
      1.0,
      0.5820000171661377
    ],
    [
      0.0,
      0.0,
      0.0,
      1.0
    ]
  ],
  "position_xyz": [
    0.07232428509296399,
    -0.060040432248501065,
    0.5820000171661377
  ],
  "quaternion_xyzw": [
    0.0,
    0.0,
    0.0,
    1.0
  ],
  "width_m": 0.05,
  "approach_dir_xyz": [
    0.0,
    0.0,
    1.0
  ],
  "source": {
    "candidate_index": 0,
    "segment_id": 0,
    "grasp_index": 0,
    "predictions_npz": null
  }
}

=== Verification ===
Position matches: True
Quaternion matches: True
Width matches: True

✓ Complete 2D-to-6D alignment pipeline verified!


## Part 8: Visualize Web AI 6-DOF Grasp Results

Paste the JSON response from the web AI (ChatGPT / Claude) below. The web AI
received RGB + depth visualizations and returned 6-DOF grasp candidates using
the prompt from `web_ai_align_prompt.md`.

This section:
1. Takes your pasted web AI JSON result
2. Loads the RGB image from the capture folder
3. Draws each candidate's grasp point, closing direction, and approach direction on the image
4. Shows the actual 3D position from the camera intrinsics

In [ ]:
# =============================================================================
# Paste the web AI JSON response below (replace the placeholder)
# =============================================================================
WEB_AI_RESULT = {
  "target": "Rubik's cube",
  "image_size": [720, 1280],
  "camera_intrinsics_used": {"fx": 640, "fy": 640, "cx": 640, "cy": 360},
  "candidates": [
    {
      "rank": 1,
      "position_3d": [0.0956, -0.0543, 1.20],
      "gripper_angle_deg": 0.0,
      "closing_direction_3d": [1.0, 0.0, 0.0],
      "approach_direction_3d": [0.0, 0.0, 1.0],
      "pixel_uv": [691, 331],
      "align_point_norm": [460, 540],
      "estimated_depth_m": 1.20,
      "estimated_object_width_along_close_m": 0.057,
      "reasoning": "Zone: Center front face. Depth color: Cyan-blue transition -> Z≈1.20m. Position computed: X=(691-640)*1.20/640=0.0956, Y=(331-360)*1.20/640=-0.0543, Z=1.20. An angle of 0.0 degrees allows the jaws to grasp the left and right sides of the cube along the X-axis. The estimated 0.057m width of a standard Rubik's cube fits perfectly within the 0.02m - 0.08m capacity. This candidate represents a standard horizontal alignment."
    },
    {
      "rank": 2,
      "position_3d": [0.0656, -0.0543, 1.20],
      "gripper_angle_deg": 90.0,
      "closing_direction_3d": [0.0, 1.0, 0.0],
      "approach_direction_3d": [0.0, 0.0, 1.0],
      "pixel_uv": [675, 331],
      "align_point_norm": [460, 527],
      "estimated_depth_m": 1.20,
      "estimated_object_width_along_close_m": 0.057,
      "reasoning": "Zone: Left-center front face. Depth color: Cyan-blue -> Z≈1.20m. Position computed: X=(675-640)*1.20/640=0.0656, Y=(331-360)*1.20/640=-0.0543, Z=1.20. Shifted ~3cm left from candidate 1 to meet diversity rules. Angle of 90.0 degrees closes along the Y-axis (top and bottom sides of the cube). This differs significantly (≥30°) from the horizontal grasp and the object width vertically (0.057m) still perfectly fits the jaw constraints."
    }
  ]
}

# =============================================================================
# Load RGB image and actual camera intrinsics
# =============================================================================
from PIL import Image, ImageDraw, ImageFont
import colorsys
import math
import numpy as np

# Use the same capture folder from Part 1, or override below
capture_dir = sample_dir  # from Part 1
rgb_pil = Image.open(capture_dir / 'color_preview.jpg').convert('RGB')
K_actual = camera_data['K'].astype(np.float64)  # actual intrinsics (not web AI's approximation)

print(f"Capture: {capture_dir}")
print(f"Image size: {rgb_pil.size}")
print(f"Actual K:\n{K_actual}")
print(f"Web AI used approximate K: {WEB_AI_RESULT['camera_intrinsics_used']}")

# =============================================================================
# Drawing helpers (adapted from grasp_server/grasp_viz.py)
# =============================================================================

def _rank_color(rank_index, total):
    """HSV sweep: blue (rank 1) → green → orange/red (last rank)."""
    hue = 0.62 - (0.55 * (rank_index / max(total, 1)))
    r, g, b = colorsys.hsv_to_rgb(hue % 1.0, 0.85, 1.0)
    return int(255 * r), int(255 * g), int(255 * b)


def _project(xyz, K):
    """Pinhole projection: 3D camera-frame point → 2D pixel (u, v)."""
    x, y, z = float(xyz[0]), float(xyz[1]), float(xyz[2])
    if z <= 0.0:
        return None
    u = K[0, 0] * x / z + K[0, 2]
    v = K[1, 1] * y / z + K[1, 2]
    return float(u), float(v)


def _draw_arrow(draw, start_xy, end_xy, color, line_width=3):
    """Draw a line with a triangular arrowhead (same algorithm as grasp_viz)."""
    draw.line([start_xy, end_xy], fill=color, width=line_width)
    sx, sy = start_xy
    ex, ey = end_xy
    dx, dy = ex - sx, ey - sy
    length = math.hypot(dx, dy)
    if length < 1e-6:
        return
    ux, uy = dx / length, dy / length       # unit vector along the line
    lx, ly = -uy, ux                         # perpendicular (left)
    head_len = max(10, min(18, length * 0.22))
    head_w = head_len * 0.45
    tip = (ex, ey)
    base = (ex - head_len * ux, ey - head_len * uy)
    left_wing = (base[0] + head_w * lx, base[1] + head_w * ly)
    right_wing = (base[0] - head_w * lx, base[1] - head_w * ly)
    draw.polygon([tip, left_wing, right_wing], fill=color)


def _try_load_font(size):
    """Load a reasonable font on macOS / Linux, falling back to PIL default."""
    import platform
    paths = []
    if platform.system() == 'Darwin':
        paths = [
            '/System/Library/Fonts/Helvetica.ttc',
            '/System/Library/Fonts/Supplemental/Arial.ttf',
            '/Library/Fonts/Arial.ttf',
        ]
    else:
        paths = [
            '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
            '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf',
        ]
    for p in paths:
        try:
            return ImageFont.truetype(p, size)
        except (IOError, OSError):
            continue
    return ImageFont.load_default()


# =============================================================================
# Main visualization function
# =============================================================================

def draw_web_ai_grasps(rgb_pil, candidates, K_actual):
    """Overlay 6-DOF grasp candidates from the web AI onto an RGB image.

    For each candidate this draws:
      - Center dot at ``pixel_uv`` (where the AI intended to grasp)
      - Closing-direction arrow (3D → projected with actual K)
      - Approach-direction arrow (+Z, desaturated colour)
      - Diagnostic cross if ``position_3d`` projects far from ``pixel_uv``
      - Rank label, depth estimate, and 3D position text
    """
    img = rgb_pil.copy()
    draw = ImageDraw.Draw(img)
    total = len(candidates)
    dot_r = max(8, min(img.width, img.height) // 80)
    font = _try_load_font(dot_r + 2)
    font_sm = _try_load_font(max(8, dot_r - 2))

    for c in candidates:
        rank = c.get('rank', 0)
        color = _rank_color(rank - 1, total)
        # Desaturated version for approach / secondary info
        light = tuple(min(255, ch + 80) for ch in color)

        # ---- centre dot ----
        pixel_uv = c.get('pixel_uv')
        if pixel_uv is None:
            print(f"  [WARN] Candidate #{rank}: missing pixel_uv, skipping")
            continue
        uc, vc = int(pixel_uv[0]), int(pixel_uv[1])
        r = dot_r
        draw.ellipse([uc - r, vc - r, uc + r, vc + r],
                     fill=color, outline='black', width=2)

        pos_3d = c.get('position_3d')
        close_dir = c.get('closing_direction_3d')
        angle_deg = c.get('gripper_angle_deg')

        # Normalise closing direction (fall back to angle_deg)
        if close_dir is not None:
            close_dir = np.array(close_dir, dtype=np.float64)
            n = np.linalg.norm(close_dir)
            if n > 1e-9:
                close_dir = (close_dir / n).tolist()
            # Clamp tiny cz to 0
            if abs(close_dir[2]) < 1e-6:
                close_dir[2] = 0.0
        elif angle_deg is not None:
            th = math.radians(angle_deg)
            close_dir = [math.cos(th), math.sin(th), 0.0]

        # ---- closing arrow (projected from 3D) ----
        if pos_3d and close_dir:
            start_uv = _project(pos_3d, K_actual)
            end_3d = [pos_3d[0] + 0.05 * close_dir[0],
                      pos_3d[1] + 0.05 * close_dir[1],
                      pos_3d[2] + 0.05 * close_dir[2]]
            end_uv = _project(end_3d, K_actual)
            if start_uv and end_uv:
                _draw_arrow(draw, start_uv, end_uv, color, line_width=3)

        # ---- approach arrow (+Z, into the scene) ----
        if pos_3d:
            start_uv = _project(pos_3d, K_actual)
            app_end_3d = [pos_3d[0], pos_3d[1], pos_3d[2] + 0.08]
            app_end_uv = _project(app_end_3d, K_actual)
            if start_uv and app_end_uv:
                _draw_arrow(draw, start_uv, app_end_uv, light, line_width=2)

        # ---- diagnostic cross: where position_3d projects vs pixel_uv ----
        if pos_3d:
            proj = _project(pos_3d, K_actual)
            if proj:
                pu, pv = int(proj[0]), int(proj[1])
                dist = math.hypot(pu - uc, pv - vc)
                if dist > 5:
                    cs = max(3, r // 2)
                    draw.line([pu - cs, pv, pu + cs, pv], fill=color, width=2)
                    draw.line([pu, pv - cs, pu, pv + cs], fill=color, width=2)

        # ---- text labels ----
        lx = uc + r + 4   # label anchor (right of dot)
        ly = vc - r        # top of label block

        rank_str = f"#{rank}"
        # shadow
        draw.text((lx + 1, ly + 1), rank_str, fill='black', font=font)
        draw.text((lx, ly), rank_str, fill=color, font=font)

        # depth + angle info
        est_z = c.get('estimated_depth_m', pos_3d[2] if pos_3d else None)
        angle_str = f"{angle_deg:.0f}°" if angle_deg is not None else "?"
        line2 = f"Z≈{est_z:.2f}m  θ={angle_str}"
        info_y = ly + font.size + 2
        draw.text((lx + 1, info_y + 1), line2, fill='black', font=font_sm)
        draw.text((lx, info_y), line2, fill=light, font=font_sm)

        # 3D position
        if pos_3d:
            line3 = f"({pos_3d[0]:.3f}, {pos_3d[1]:.3f}, {pos_3d[2]:.3f}) m"
            pos_y = info_y + font_sm.size + 2
            draw.text((lx + 1, pos_y + 1), line3, fill='black', font=font_sm)
            draw.text((lx, pos_y), line3, fill=light, font=font_sm)

    return img


# =============================================================================
# Run
# =============================================================================
candidates = WEB_AI_RESULT.get('candidates', [])
if not candidates:
    print('ERROR: no candidates found in WEB_AI_RESULT. Did you paste the JSON?')
else:
    print(f'Drawing {len(candidates)} candidate(s) for "{WEB_AI_RESULT.get("target", "?")}" ...')
    annotated = draw_web_ai_grasps(rgb_pil, candidates, K_actual)

    # Save next to the capture
    out_path = capture_dir / 'web_ai_grasp_viz.jpg'
    annotated.save(out_path, quality=92)
    print(f'Saved: {out_path}')

    # Display inline
    try:
        from IPython.display import display as ipy_display
        ipy_display(annotated)
    except ImportError:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(14, 8))
        plt.imshow(annotated)
        plt.axis('off')
        plt.title(f'Web AI 6-DOF Grasps — {WEB_AI_RESULT.get("target", "")}')
        plt.tight_layout()
        plt.show()

In [ ]:
# Optional: show side-by-side with depth preview + compare estimated vs actual depth
import numpy as np
from vg_pipeline.align import sample_depth_median

show_depth_side_by_side = True   # set to False to skip

if show_depth_side_by_side:
    depth_pil = Image.open(capture_dir / 'depth_preview.jpg').convert('RGB')
    # Resize depth to match annotated height
    depth_resized = depth_pil.resize((annotated.width, annotated.height))
    combined = Image.new('RGB', (annotated.width * 2, annotated.height))
    combined.paste(annotated, (0, 0))
    combined.paste(depth_resized, (annotated.width, 0))

    # Also mark the grasp points on the depth image
    draw_d = ImageDraw.Draw(combined)
    for c in candidates:
        pixel_uv = c.get('pixel_uv')
        if pixel_uv is None:
            continue
        rank = c.get('rank', 0)
        color = _rank_color(rank - 1, len(candidates))
        uc, vc = int(pixel_uv[0]), int(pixel_uv[1])
        # Draw on the depth side (offset by annotated.width)
        r = max(6, min(annotated.width, annotated.height) // 100)
        du = annotated.width + uc
        dv = vc
        draw_d.ellipse([du - r, dv - r, du + r, dv + r], fill=color, outline='white', width=2)
        draw_d.text((du + r + 2, dv - r), f"#{rank}", fill=color)

    display(combined) if 'ipy_display' in dir() else None
    print("Left: RGB with 6-DOF grasps  |  Right: Depth preview with grasp points\n")

# ---- Depth comparison table ----
print(f"{'Rank':<6} {'pixel_uv':<14} {'AI est Z':<10} {'Actual Z (5x5 median)':<24} {'Error (cm)':<12}")
print("-" * 68)
depth_map = camera_data['depth'].astype(np.float64)
for c in candidates:
    rank = c.get('rank', '?')
    pixel_uv = c.get('pixel_uv')
    est_z = c.get('estimated_depth_m', float('nan'))
    if pixel_uv:
        try:
            actual_z = sample_depth_median(depth_map, int(pixel_uv[1]), int(pixel_uv[0]), window=5)
            error_cm = (est_z - actual_z) * 100
            print(f"{rank:<6} {str(pixel_uv):<14} {est_z:<10.3f} {actual_z:<24.4f} {error_cm:+<12.1f}")
        except ValueError:
            print(f"{rank:<6} {str(pixel_uv):<14} {est_z:<10.3f} {'(no valid depth)':<24} {'N/A':<12}")
    else:
        print(f"{rank:<6} {'(missing)':<14} {est_z:<10.3f} {'N/A':<24} {'N/A':<12}")

print("\nNote: 'AI est Z' is from the web AI reading the JET colormap (approximate).")
print("'Actual Z' is sampled from the raw float32 depth map (millimeter-accurate).")
print("Large errors mean the web AI misread the depth color, or the intrinsic mismatch was significant.")

## Part 9: Interactive 3D Grasp Visualization

This section creates an **interactive 3D plot** (powered by Plotly) showing:

- The full scene as a colored **point cloud** (back-projected from depth + RGB)
- Each grasp candidate as a **gripper wireframe** with closing/approach arrows
- **XYZ axis indicators** per grasp: red=closing, green=lateral, blue=approach
- Comparison markers showing the web AI's estimated vs actual 3D position

Drag to rotate, scroll to zoom, hover for details. Reuses `WEB_AI_RESULT`
and `candidates` from Part 8.

In [ ]:
# =============================================================================
# 3-D interactive grasp visualization using Plotly
# Reuses: rgb_pil, candidates, WEB_AI_RESULT, _rank_color from Part 8
#         camera_data, sample_dir from Part 1
# =============================================================================

import plotly.graph_objects as go
import numpy as np
from vg_pipeline.geometry import backproject_depth_with_mask
from vg_pipeline.align import sample_depth_median, deproject_pixel
from vg_pipeline.grasp_results import FINGER_LENGTH_METERS, GRIPPER_DEPTH_METERS

# ── 1. Back-project the scene point cloud ──────────────────────────────────
depth_map = camera_data['depth'].astype(np.float64)
K_actual = camera_data['K'].astype(np.float64)

# Load RGB as numpy for colour sampling
rgb_np = np.asarray(rgb_pil)  # (H, W, 3) uint8
valid_mask = np.isfinite(depth_map) & (depth_map > 0.0)

print("Back-projecting depth map to point cloud ...")
pts_full = backproject_depth_with_mask(depth_map, K_actual)  # (N, 3) float32
colors_full = rgb_np[valid_mask]                               # (N, 3) uint8
print(f"  Full point cloud: {pts_full.shape[0]:,} points")

# Subsample for smooth interactivity
MAX_PTS = 15000
if pts_full.shape[0] > MAX_PTS:
    idx = np.random.choice(pts_full.shape[0], MAX_PTS, replace=False)
    pts = pts_full[idx]
    cols = colors_full[idx]
    print(f"  Downsampled to {MAX_PTS:,} points for rendering")
else:
    pts = pts_full
    cols = colors_full

# ── 2. Gripper wireframe helper (replicates grasp_viz._gripper_wireframe) ──
def _gripper_wireframe(pose_4x4, width_m):
    """Return (N,3) world points + list of (start, end) edge index pairs."""
    w = width_m if (width_m is not None and width_m > 0) else 0.08
    R = pose_4x4[:3, :3]
    t = pose_4x4[:3, 3]
    local = np.array([
        [0.0,     0.0, 0.0],                                    # 0: palm center
        [0.0,     0.0, GRIPPER_DEPTH_METERS],                   # 1: palm base
        [-w / 2,  0.0, GRIPPER_DEPTH_METERS],                   # 2: left base
        [-w / 2,  0.0, GRIPPER_DEPTH_METERS + FINGER_LENGTH_METERS],  # 3: left tip
        [ w / 2,  0.0, GRIPPER_DEPTH_METERS],                   # 4: right base
        [ w / 2,  0.0, GRIPPER_DEPTH_METERS + FINGER_LENGTH_METERS],  # 5: right tip
    ], dtype=np.float32)
    world = (local @ R.T) + t.reshape(1, 3)
    edges = [(0, 1), (2, 3), (4, 5), (2, 4)]  # palm→base, left finger, right finger, cross-bar
    return world, edges


# ── 3. Build per-candidate data: actual 3D position + 4×4 pose ────────────
candidates_3d = []

for c in candidates:
    pixel_uv = c.get('pixel_uv')
    if pixel_uv is None:
        print(f"  [WARN] Candidate #{c.get('rank')}: missing pixel_uv, skipping")
        continue

    u, v = int(pixel_uv[0]), int(pixel_uv[1])

    # Sample actual depth at the AI-chosen pixel
    try:
        z_actual = sample_depth_median(depth_map, v, u, window=5)
    except ValueError:
        print(f"  [WARN] Candidate #{c.get('rank')}: no valid depth at ({u}, {v}), skipping")
        continue

    # Deproject using actual intrinsics (not the web AI's approximation)
    pos_actual = deproject_pixel(u, v, z_actual, K_actual)

    # Closing direction from web AI (fall back to angle_deg)
    close_dir = c.get('closing_direction_3d')
    angle_deg = c.get('gripper_angle_deg')
    if close_dir is not None:
        close_dir = np.array(close_dir, dtype=np.float64)
        n = np.linalg.norm(close_dir)
        if n > 1e-9:
            close_dir = close_dir / n
        if abs(close_dir[2]) < 1e-6:
            close_dir[2] = 0.0
    elif angle_deg is not None:
        th = np.radians(angle_deg)
        close_dir = np.array([np.cos(th), np.sin(th), 0.0], dtype=np.float64)
    else:
        print(f"  [WARN] Candidate #{c.get('rank')}: no closing direction or angle, skipping")
        continue

    approach = np.array([0.0, 0.0, 1.0], dtype=np.float64)
    lateral = np.cross(approach, close_dir)
    lat_norm = np.linalg.norm(lateral)
    if lat_norm < 1e-9:
        lateral = np.array([1.0, 0.0, 0.0], dtype=np.float64)
    else:
        lateral = lateral / lat_norm

    # Build 4×4 pose matrix (camera frame)
    pose = np.eye(4, dtype=np.float64)
    pose[:3, 0] = close_dir       # closing / base
    pose[:3, 1] = lateral         # lateral
    pose[:3, 2] = approach        # approach (+Z)
    pose[:3, 3] = pos_actual      # 3D position from actual depth

    ai_pos = c.get('position_3d')

    candidates_3d.append({
        **c,
        'pos_actual': pos_actual,
        'z_actual': z_actual,
        'pose': pose,
        'close_dir': close_dir,
        'ai_pos_3d': np.array(ai_pos, dtype=np.float64) if ai_pos else None,
    })

    dist_mm = 0.0
    if ai_pos is not None:
        dist_mm = np.linalg.norm(np.array(ai_pos) - pos_actual) * 1000
    print(f"  #{c.get('rank')}: pixel=({u},{v})  Z_actual={z_actual:.3f}m  "
          f"pos={pos_actual.round(4)}  AI_pos_offset={dist_mm:.1f}mm")

if not candidates_3d:
    raise RuntimeError("No valid candidates for 3D visualization. Check pixel_uv values.")

print(f"\nReady to render {len(candidates_3d)} grasp(s) in 3D.")

# ── 4. Build Plotly figure ────────────────────────────────────────────────
fig = go.Figure()

# ---- Scene point cloud ----
color_strs = [f'rgb({r},{g},{b})' for r, g, b in cols]
fig.add_trace(go.Scatter3d(
    x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
    mode='markers',
    name='scene point cloud',
    marker={'size': 2, 'opacity': 0.6, 'color': color_strs},
    hoverinfo='skip',
))

# ---- Per-candidate artifacts ----
AXIS_LENGTH = 0.05  # metres

for i, c in enumerate(candidates_3d):
    rank = c['rank']
    color = _rank_color(i, len(candidates_3d))
    rgb_str = f'rgb({color[0]},{color[1]},{color[2]})'
    # Semi-transparent version for secondary elements
    rgb_faint = f'rgba({color[0]},{color[1]},{color[2]},0.45)'

    pose = c['pose']
    center = c['pos_actual']
    rotation = pose[:3, :3]
    width_m = c.get('estimated_object_width_along_close_m', 0.05)

    # -- Gripper wireframe --
    wf_pts, wf_edges = _gripper_wireframe(pose, width_m)
    for s_idx, e_idx in wf_edges:
        seg = wf_pts[[s_idx, e_idx]]
        fig.add_trace(go.Scatter3d(
            x=seg[:, 0], y=seg[:, 1], z=seg[:, 2],
            mode='lines',
            line={'color': rgb_str, 'width': 6},
            hoverinfo='skip',
            showlegend=False,
        ))

    # -- XYZ axis arrows --
    AXIS_SPECS = [
        ('X closing',  0, 'rgb(255,90,90)'),
        ('Y lateral',  1, 'rgb(90,220,100)'),
        ('Z approach', 2, 'rgb(80,150,255)'),
    ]
    for ax_name, col_idx, ax_color in AXIS_SPECS:
        tip = center + AXIS_LENGTH * rotation[:, col_idx]
        fig.add_trace(go.Scatter3d(
            x=[center[0], tip[0]],
            y=[center[1], tip[1]],
            z=[center[2], tip[2]],
            mode='lines',
            name=ax_name if i == 0 else None,
            line={'color': ax_color, 'width': 5},
            hoverinfo='skip',
            showlegend=(i == 0),
        ))

    # -- Center marker with rank label --
    angle_str = f"{c.get('gripper_angle_deg', '?'):.0f}°"
    fig.add_trace(go.Scatter3d(
        x=[center[0]], y=[center[1]], z=[center[2]],
        mode='markers+text',
        name=f"#{rank}",
        text=[f"#{rank}"],
        textposition='top center',
        textfont={'size': 12, 'color': rgb_str},
        marker={'size': 8, 'color': rgb_str, 'line': {'color': 'black', 'width': 1}},
        hovertemplate=(
            f"<b>#{rank}</b><br>"
            f"pos=({center[0]:.3f}, {center[1]:.3f}, {center[2]:.3f}) m<br>"
            f"Z={c['z_actual']:.3f} m<br>"
            f"θ={angle_str}<br>"
            f"width≈{width_m*1000:.0f} mm"
            f"<extra></extra>"
        ),
    ))

    # -- Web AI's estimated position (if differs by > 1 cm) --
    ai_pos = c.get('ai_pos_3d')
    if ai_pos is not None:
        dist_m = np.linalg.norm(ai_pos - center)
        if dist_m > 0.01:
            fig.add_trace(go.Scatter3d(
                x=[ai_pos[0]], y=[ai_pos[1]], z=[ai_pos[2]],
                mode='markers',
                marker={'size': 5, 'color': rgb_str, 'symbol': 'diamond-open', 'line': {'width': 2}},
                name=f"AI est #{rank}" if i == 0 else None,
                showlegend=(i == 0),
                hovertemplate=(
                    f"<b>AI estimated #{rank}</b><br>"
                    f"offset from actual: {dist_m*100:.1f} cm<br>"
                    f"position=({ai_pos[0]:.3f}, {ai_pos[1]:.3f}, {ai_pos[2]:.3f})"
                    f"<extra></extra>"
                ),
            ))
            # Dashed connector line from AI estimate to actual position
            fig.add_trace(go.Scatter3d(
                x=[ai_pos[0], center[0]],
                y=[ai_pos[1], center[1]],
                z=[ai_pos[2], center[2]],
                mode='lines',
                line={'color': rgb_faint, 'width': 1, 'dash': 'dot'},
                hoverinfo='skip',
                showlegend=False,
            ))

# ── 5. Layout & display ───────────────────────────────────────────────────
fig.update_layout(
    title=(
        f"<b>3D Grasp Visualization</b> — {WEB_AI_RESULT.get('target', '')}<br>"
        f"<sup>XYZ axes per grasp: "
        f"<span style='color:#ff5a5a'>red=closing</span>  "
        f"<span style='color:#5adc64'>green=lateral</span>  "
        f"<span style='color:#5096ff'>blue=approach (+Z)</span></sup>"
    ),
    margin={'l': 0, 'r': 0, 't': 80, 'b': 0},
    scene={
        'xaxis_title': 'X (m) → right',
        'yaxis_title': 'Y (m) → down',
        'zaxis_title': 'Z (m) → into scene',
        'aspectmode': 'data',
    },
    legend={'orientation': 'h', 'yanchor': 'bottom', 'y': -0.15},
    hovermode='closest',
)

# Inline display (works in Jupyter / VSCode notebooks)
fig.show()

# Also save as self-contained HTML
html_path = capture_dir / 'web_ai_grasp_viz_3d.html'
fig.write_html(str(html_path), include_plotlyjs=True, full_html=True)
print(f"\nSaved interactive 3D view: {html_path}")